In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt

# 1. Open the file in read-only mode
file_path = '/kaggle/input/stead-chunk-2/chunk2.hdf5'
with h5py.File(file_path, 'r') as f:
    # 2. Inspect the high-level groups
    print("Keys in the HDF5 file:", list(f.keys()))
    
    # 3. Access a specific trace (using an example from your CSV data)
    # Replace '109C.TA_20060723155859_EV' with a trace_name from your CSV
    trace_name = '109C.TA_20060723155859_EV'
    dataset = f[f'data/{trace_name}']
    
    # 4. Convert to numpy and check metadata
    data = np.array(dataset)
    print(f"Shape of {trace_name}: {data.shape}")
    
    # 5. Extract specific attributes (Metadata stored inside the HDF5)
    for attr in dataset.attrs:
        print(f"{attr}: {dataset.attrs[attr]}")

In [ ]:
!rm -rf /kaggle/working/*

In [ ]:
# ============================================
# BLOCK 1: IMPORTS AND CONFIGURATION (FIXED)
# ============================================

import os
import numpy as np
import pandas as pd
import h5py
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import autocast, GradScaler
from tqdm import tqdm
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d
from scipy.signal import find_peaks
from sklearn.metrics import precision_recall_curve, average_precision_score
import warnings
import gc
warnings.filterwarnings('ignore')

# ============================================
# BLOCK 1: CONFIGURATION (updated with depth normalization)
# ============================================

class Config:
    # Paths
    csv_path = "/kaggle/input/stead-chunk-2/chunk2.csv"
    hdf5_path = "/kaggle/input/stead-chunk-2/chunk2.hdf5"
    output_dir = "/kaggle/working/output"
    checkpoint_dir = "/kaggle/working/checkpoints"
    
    # Data parameters
    sampling_rate = 100  # Hz
    window_sec = 20
    window_samples = window_sec * sampling_rate
    p_label_width = 1.0
    s_label_width = 1.0
    event_label_width = 2.0
    max_stations_per_event = 8
    
    # Model architecture
    backbone_out_channels = 512
    phase_hidden = 32
    attention_heads = 4
    attention_hidden = 128
    event_hidden = 64
    depth_hidden = 32
    
    # Training
    batch_size = 2
    epochs = 10
    lr = 1e-3
    weight_decay = 1e-4
    num_workers = 0
    device = "cuda" if torch.cuda.is_available() else "cpu"
    mixed_precision = False  # Disable for debugging
    gradient_clip = 1.0
    
    # Loss weights
    lambda_p = 1.0
    lambda_s = 1.0
    lambda_event = 1.0
    lambda_depth = 0.1
    
    # Depth normalization
    depth_mean = 10.83
    depth_std = 7.08
    
    # Dataset split
    train_ratio = 0.8
    val_ratio = 0.1
    test_ratio = 0.1
    seed = 42
    max_events = 2000
    
    # Inference
    detection_threshold = 0.5
    min_stations = 3
    nms_window_sec = 5.0

# Create directories
os.makedirs(Config.output_dir, exist_ok=True)
os.makedirs(Config.checkpoint_dir, exist_ok=True)

print(f"Configuration loaded")
print(f"Using device: {Config.device}")
print(f"Window samples: {Config.window_samples}")
print(f"Batch size: {Config.batch_size}")
print(f"Num workers: {Config.num_workers}")

In [ ]:
# ============================================
# BLOCK 2: UTILITY FUNCTIONS (with gaussian_label fix)
# ============================================

def gaussian_label(length, peak_idx, sigma_sec, sr):
    """Create 1D Gaussian label centered at peak_idx."""
    x = np.arange(length)
    sigma = sigma_sec * sr
    if sigma == 0:
        sigma = 1  # Avoid division by zero
    label = np.exp(-0.5 * ((x - peak_idx) / sigma)**2)
    return label

def normalize_waveform(waveform):
    """Normalize each component independently."""
    mean = waveform.mean(axis=1, keepdims=True)
    std = waveform.std(axis=1, keepdims=True) + 1e-6
    return (waveform - mean) / std

def haversine_distance(lat1, lon1, lat2, lon2):
    """Vectorized haversine distance in km."""
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

def count_parameters(model):
    """Count trainable parameters."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [ ]:
# ============================================
# BLOCK 3: DATASET CLASS (with depth normalization)
# ============================================

class STEADMultiStationDataset(Dataset):
    """
    Groups waveforms by event (source_id).
    Fixed version with proper shape handling and depth normalization.
    """
    def __init__(self, csv_path, hdf5_path, split='train', config=Config):
        self.config = config
        print(f"Loading CSV: {csv_path}")
        
        # Load only necessary columns
        self.df = pd.read_csv(csv_path, 
                              usecols=['source_id', 'trace_name', 'trace_start_time', 
                                      'source_origin_time', 'source_latitude', 'source_longitude',
                                      'source_depth_km', 'receiver_latitude', 'receiver_longitude',
                                      'p_arrival_sample', 's_arrival_sample'],
                              low_memory=False)
        
        # Convert numeric columns properly
        numeric_cols = ['source_latitude', 'source_longitude', 'source_depth_km',
                       'receiver_latitude', 'receiver_longitude',
                       'p_arrival_sample', 's_arrival_sample']
        
        for col in numeric_cols:
            self.df[col] = pd.to_numeric(self.df[col], errors='coerce')
        
        # Keep only events with both P and S picks
        self.df = self.df.dropna(subset=['p_arrival_sample', 's_arrival_sample']).reset_index(drop=True)
        
        # Filter out unrealistic depths (negative or too deep)
        self.df = self.df[self.df['source_depth_km'].between(0, 50)].reset_index(drop=True)
        
        # Group by event
        self.event_groups = self.df.groupby('source_id')
        self.event_ids = list(self.event_groups.groups.keys())
        print(f"Total events: {len(self.event_ids)}")
        
        # Limit events if specified
        if config.max_events is not None and len(self.event_ids) > config.max_events:
            np.random.seed(config.seed)
            self.event_ids = list(np.random.choice(self.event_ids, config.max_events, replace=False))
            print(f"Limited to {config.max_events} events")
        
        # Split
        np.random.seed(config.seed)
        np.random.shuffle(self.event_ids)
        n = len(self.event_ids)
        n_train = int(n * config.train_ratio)
        n_val = int(n * config.val_ratio)
        
        if split == 'train':
            self.event_ids = self.event_ids[:n_train]
        elif split == 'val':
            self.event_ids = self.event_ids[n_train:n_train+n_val]
        else:
            self.event_ids = self.event_ids[n_train+n_val:]
        
        print(f"{split} set size: {len(self.event_ids)} events")
        
        # Open HDF5 file
        self.hdf5_file = h5py.File(hdf5_path, 'r')
        
    def __len__(self):
        return len(self.event_ids)
    
    def __getitem__(self, idx):
        event_id = self.event_ids[idx]
        rows = self.event_groups.get_group(event_id)
        
        # Sample stations if too many
        if len(rows) > self.config.max_stations_per_event:
            rows = rows.sample(n=self.config.max_stations_per_event, random_state=idx)
        
        n_stations = len(rows)
        T = self.config.window_samples
        half = T // 2
        
        waveforms = []
        station_coords = []
        p_pick_samples = []
        s_pick_samples = []
        
        # Event info
        try:
            event_lat = float(rows.iloc[0]['source_latitude'])
            event_lon = float(rows.iloc[0]['source_longitude'])
            event_depth = float(rows.iloc[0]['source_depth_km'])
            # Normalize depth
            event_depth_norm = (event_depth - self.config.depth_mean) / self.config.depth_std
            source_time = pd.to_datetime(rows.iloc[0]['source_origin_time'])
        except (ValueError, TypeError) as e:
            return self.__getitem__((idx + 1) % len(self.event_ids))
        
        for _, row in rows.iterrows():
            trace_name = row['trace_name']
            try:
                data = self.hdf5_file[f'data/{trace_name}'][()]  # Should be (3, samples)
                
                # Check if data is transposed (samples, 3) and fix if needed
                if data.shape[0] != 3 and data.shape[1] == 3:
                    data = data.T  # Transpose to (3, samples)
                
                # Get receiver coordinates
                try:
                    rec_lat = float(row['receiver_latitude'])
                    rec_lon = float(row['receiver_longitude'])
                except (ValueError, TypeError):
                    continue
                
                trace_start = pd.to_datetime(row['trace_start_time'])
                origin_sec = (source_time - trace_start).total_seconds()
                start_sample = int(origin_sec - half)
                end_sample = int(origin_sec + half)
                
                # Get pick samples
                try:
                    p_pick = float(row['p_arrival_sample'])
                    s_pick = float(row['s_arrival_sample'])
                except (ValueError, TypeError):
                    continue
                
                # Handle trace boundaries
                if start_sample < 0 or end_sample > data.shape[1]:
                    pad_left = max(0, -start_sample)
                    pad_right = max(0, end_sample - data.shape[1])
                    data_padded = np.pad(data, ((0,0), (pad_left, pad_right)), mode='constant')
                    w = data_padded[:, start_sample+pad_left : start_sample+pad_left+T]
                    p_pick = p_pick + pad_left - start_sample
                    s_pick = s_pick + pad_left - start_sample
                else:
                    w = data[:, start_sample:end_sample]
                    p_pick = p_pick - start_sample
                    s_pick = s_pick - start_sample
                
                # Ensure correct length
                if w.shape[1] > T:
                    w = w[:, :T]
                elif w.shape[1] < T:
                    pad = T - w.shape[1]
                    w = np.pad(w, ((0,0), (0, pad)), mode='constant')
                
                # Normalize each component (robust normalization)
                for comp in range(3):
                    comp_data = w[comp]
                    median = np.median(comp_data)
                    q1 = np.percentile(comp_data, 25)
                    q3 = np.percentile(comp_data, 75)
                    iqr = q3 - q1 + 1e-6
                    w[comp] = (comp_data - median) / iqr
                
                waveforms.append(w)  # Each w is (3, T)
                station_coords.append([rec_lat, rec_lon])
                p_pick_samples.append(p_pick)
                s_pick_samples.append(s_pick)
                
            except Exception as e:
                continue
        
        # If no valid stations, try next event
        if len(waveforms) == 0:
            return self.__getitem__((idx + 1) % len(self.event_ids))
        
        n_stations = len(waveforms)
        
        # Stack waveforms - should be (n_stations, 3, T)
        try:
            waveforms = np.stack(waveforms)
            station_coords = np.array(station_coords)
            p_pick = np.array(p_pick_samples)
            s_pick = np.array(s_pick_samples)
        except Exception as e:
            print(f"Error stacking: {e}")
            return self.__getitem__((idx + 1) % len(self.event_ids))
        
        # Create Gaussian labels (original resolution)
        p_label = np.zeros((n_stations, T))
        s_label = np.zeros((n_stations, T))
        for i in range(n_stations):
            if 0 <= p_pick[i] < T:
                p_label[i] = gaussian_label(T, int(p_pick[i]), self.config.p_label_width, self.config.sampling_rate)
            if 0 <= s_pick[i] < T:
                s_label[i] = gaussian_label(T, int(s_pick[i]), self.config.s_label_width, self.config.sampling_rate)
        
        # Event time label (original resolution)
        event_time_label = gaussian_label(T, half, self.config.event_label_width, self.config.sampling_rate)
        
        # Convert to tensors
        waveforms = torch.FloatTensor(waveforms)  # (n_stations, 3, T)
        station_coords = torch.FloatTensor(station_coords)
        p_label = torch.FloatTensor(p_label)  # (n_stations, T)
        s_label = torch.FloatTensor(s_label)  # (n_stations, T)
        event_time_label = torch.FloatTensor(event_time_label)  # (T,)
        mask = torch.ones(n_stations)
        event_depth_tensor = torch.FloatTensor([event_depth_norm])  # Normalized depth
        
        # Pad to max_stations
        if n_stations < self.config.max_stations_per_event:
            pad = self.config.max_stations_per_event - n_stations
            
            # Pad waveforms: (pad, 0, 0) for station dimension
            waveforms = F.pad(waveforms, (0, 0, 0, 0, 0, pad))
            station_coords = F.pad(station_coords, (0, 0, 0, pad))
            p_label = F.pad(p_label, (0, 0, 0, pad))
            s_label = F.pad(s_label, (0, 0, 0, pad))
            mask = F.pad(mask, (0, pad))
        
        # Final shape verification
        assert waveforms.shape == (self.config.max_stations_per_event, 3, T), \
            f"Expected {(self.config.max_stations_per_event, 3, T)}, got {waveforms.shape}"
        
        return {
            'waveforms': waveforms,  # (max_stations, 3, T)
            'station_coords': station_coords,  # (max_stations, 2)
            'p_label': p_label,  # (max_stations, T)
            's_label': s_label,  # (max_stations, T)
            'event_time_label': event_time_label,  # (T,)
            'event_depth': event_depth_tensor,  # (1,) - normalized
            'mask': mask,  # (max_stations,)
            'n_stations': n_stations
        }
    
    def __del__(self):
        if hasattr(self, 'hdf5_file'):
            self.hdf5_file.close()

In [ ]:
# ============================================
# BLOCK 4: MODEL COMPONENTS
# ============================================

class ResBlock1D(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, downsample=None):
        super().__init__()
        self.conv1 = nn.Conv1d(in_ch, out_ch, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm1d(out_ch)
        self.conv2 = nn.Conv1d(out_ch, out_ch, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm1d(out_ch)
        self.downsample = downsample
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        identity = x
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        if self.downsample is not None:
            identity = self.downsample(x)
        out += identity
        out = self.relu(out)
        return out

class ResNet18_1D(nn.Module):
    def __init__(self, in_channels=3):
        super().__init__()
        self.in_ch = 64
        self.conv1 = nn.Conv1d(in_channels, 64, 7, 2, 3, bias=False)
        self.bn1 = nn.BatchNorm1d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool1d(3, 2, 1)

        self.layer1 = self._make_layer(64, 64, 2, stride=1)
        self.layer2 = self._make_layer(64, 128, 2, stride=2)
        self.layer3 = self._make_layer(128, 256, 2, stride=2)
        self.layer4 = self._make_layer(256, 512, 2, stride=2)

    def _make_layer(self, in_ch, out_ch, blocks, stride):
        downsample = None
        if stride != 1 or in_ch != out_ch:
            downsample = nn.Sequential(
                nn.Conv1d(in_ch, out_ch, 1, stride, bias=False),
                nn.BatchNorm1d(out_ch),
            )
        layers = [ResBlock1D(in_ch, out_ch, stride, downsample)]
        for _ in range(1, blocks):
            layers.append(ResBlock1D(out_ch, out_ch))
        return nn.Sequential(*layers)

    def forward(self, x):
        # x shape: (batch*stations, channels, time)
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        return x  # (batch*stations, 512, time/32)


# ============================================
# BLOCK 4.2: Phase Picking Heads (without sigmoid)
# ============================================

class PhasePicker(nn.Module):
    def __init__(self, in_channels=512, hidden=32):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, hidden, 3, padding=1)
        self.bn1 = nn.BatchNorm1d(hidden)
        self.conv2 = nn.Conv1d(hidden, hidden, 3, padding=1)
        self.bn2 = nn.BatchNorm1d(hidden)
        self.conv3 = nn.Conv1d(hidden, 1, 1)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.relu(self.bn2(self.conv2(x)))
        x = self.conv3(x).squeeze(1)
        return x  # Return logits, no sigmoid

# 4.3: Cross-Station Attention Module (NOVEL CONTRIBUTION)
# ============================================
# BLOCK 4.3: Cross-Station Attention Module (FIXED)
# ============================================

class CrossStationAttention(nn.Module):
    """
    Aggregates per-station features across stations using multi-head attention.
    Input: (batch, stations, channels, time)
    Output: (batch, channels, time)  (aggregated feature)
    """
    def __init__(self, in_channels, num_heads=4, hidden=128):
        super().__init__()
        self.in_channels = in_channels
        self.num_heads = num_heads
        self.hidden = hidden
        
        # Project each station's time steps to queries, keys, values
        self.query_proj = nn.Linear(in_channels, hidden)
        self.key_proj = nn.Linear(in_channels, hidden)
        self.value_proj = nn.Linear(in_channels, hidden)
        
        self.out_proj = nn.Linear(hidden, in_channels)
        self.norm = nn.LayerNorm(in_channels)
        self.dropout = nn.Dropout(0.1)
        
    def forward(self, x, mask=None):
        # x: (B, S, C, T)
        B, S, C, T = x.shape
        
        # Reshape to (B*T, S, C) - treat each time step independently
        x_perm = x.permute(0, 3, 1, 2).contiguous()  # (B, T, S, C)
        x_flat = x_perm.view(B*T, S, C)               # (B*T, S, C)
        
        # Project
        Q = self.query_proj(x_flat)  # (B*T, S, H)
        K = self.key_proj(x_flat)    # (B*T, S, H)
        V = self.value_proj(x_flat)  # (B*T, S, H)
        
        # Multi-head attention
        H = self.num_heads
        head_dim = self.hidden // H
        Q = Q.view(B*T, S, H, head_dim).transpose(1, 2)  # (B*T, H, S, head_dim)
        K = K.view(B*T, S, H, head_dim).transpose(1, 2)
        V = V.view(B*T, S, H, head_dim).transpose(1, 2)
        
        # Attention scores
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (head_dim ** 0.5)  # (B*T, H, S, S)
        
        if mask is not None:
            # mask: (B, S) -> expand to (B*T, H, S, S)
            mask_exp = mask.unsqueeze(1).unsqueeze(2).unsqueeze(3)  # (B, 1, 1, 1, S)
            mask_exp = mask_exp.expand(-1, T, H, S, -1)  # (B, T, H, S, S)
            mask_exp = mask_exp.reshape(B*T, H, S, S)  # (B*T, H, S, S)
            scores = scores.masked_fill(mask_exp == 0, float('-inf'))
        
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        
        # Apply attention
        out = torch.matmul(attn, V)  # (B*T, H, S, head_dim)
        out = out.transpose(1, 2).contiguous().view(B*T, S, self.hidden)  # (B*T, S, H)
        
        # Output projection
        out = self.out_proj(out)  # (B*T, S, C)
        
        # Aggregate over stations (weighted mean with mask)
        if mask is not None:
            mask_agg = mask.unsqueeze(1).expand(-1, T, -1).reshape(B*T, S, 1)  # (B*T, S, 1)
            out = (out * mask_agg).sum(dim=1) / (mask_agg.sum(dim=1) + 1e-8)
        else:
            out = out.mean(dim=1)  # (B*T, C)
        
        # Reshape back to (B, C, T)
        out = out.view(B, T, C).permute(0, 2, 1)  # (B, C, T)
        
        # Residual connection with mean of input
        identity = x.mean(dim=1)  # (B, C, T)
        out = out + identity
        out = self.norm(out.permute(0,2,1)).permute(0,2,1)
        
        return out


# ============================================
# BLOCK 4.4: Event Detection Head (without sigmoid)
# ============================================

class EventDetector(nn.Module):
    def __init__(self, in_channels=512, hidden=64):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, hidden, 3, padding=1)
        self.bn1 = nn.BatchNorm1d(hidden)
        self.conv2 = nn.Conv1d(hidden, hidden, 3, padding=1)
        self.bn2 = nn.BatchNorm1d(hidden)
        self.conv3 = nn.Conv1d(hidden, 1, 1)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.relu(self.bn2(self.conv2(x)))
        x = self.conv3(x).squeeze(1)
        return x  # Return logits, no sigmoid


# 4.5: Depth Estimation Head (NOVEL CONTRIBUTION)
class DepthEstimator(nn.Module):
    def __init__(self, in_channels=512, hidden=32):
        super().__init__()
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.fc1 = nn.Linear(in_channels, hidden)
        self.fc2 = nn.Linear(hidden, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        # x: (B, C, T)
        x = self.global_pool(x).squeeze(-1)  # (B, C)
        x = self.relu(self.fc1(x))
        x = self.fc2(x).squeeze(-1)          # (B,)
        return x

In [ ]:
# ============================================
# BLOCK 5: FULL EQNET+ MODEL
# ============================================

# ============================================
# BLOCK 5: FULL EQNET+ MODEL (FIXED)
# ============================================

class EQNetPlus(nn.Module):
    """
    EQNet+: Attention-based Multi-Station Earthquake Detection with Depth Estimation
    """
    def __init__(self, config=Config):
        super().__init__()
        self.config = config
        self.max_stations = config.max_stations_per_event
        
        # Backbone
        self.backbone = ResNet18_1D(in_channels=3)
        backbone_out = 512
        
        # Phase picking heads
        half = backbone_out // 2
        self.phase_picker_p = PhasePicker(in_channels=half, hidden=config.phase_hidden)
        self.phase_picker_s = PhasePicker(in_channels=half, hidden=config.phase_hidden)
        
        # Cross-station attention
        self.attention = CrossStationAttention(
            in_channels=backbone_out,
            num_heads=config.attention_heads,
            hidden=config.attention_hidden
        )
        
        # Event detector
        self.event_detector = EventDetector(in_channels=backbone_out, hidden=config.event_hidden)
        
        # Depth estimator
        self.depth_estimator = DepthEstimator(in_channels=backbone_out, hidden=config.depth_hidden)
        
        print(f"EQNet+ created with {count_parameters(self):,} parameters")
        
    def forward(self, waveforms, station_coords, mask):
        """
        Args:
            waveforms: (B, S, 3, T) - batch, stations, channels, time
            station_coords: (B, S, 2) - not used in this version
            mask: (B, S) - 1 for valid stations
        Returns:
            p_pred: (B, S, T')   (T' = T/32)
            s_pred: (B, S, T')
            event_pred: (B, T')
            depth_pred: (B,)
        """
        B, S, C, T = waveforms.shape
        
        # Reshape to combine batch and stations for backbone
        # Input to backbone should be (B*S, C, T)
        w = waveforms.view(B * S, C, T)
        
        # Extract features
        features = self.backbone(w)  # (B*S, 512, T')
        _, Cf, Tf = features.shape
        
        # Reshape back to separate batch and stations
        features = features.view(B, S, Cf, Tf)  # (B, S, 512, T')
        
        # Apply cross-station attention
        agg_features = self.attention(features, mask)  # (B, 512, T')
        
        # Split features for phase picking (use half for P, half for S)
        half = Cf // 2
        features_p = features[:, :, :half, :]  # (B, S, half, Tf)
        features_s = features[:, :, half:, :]  # (B, S, Cf-half, Tf)
        
        # Phase predictions - reshape to (B*S, half, Tf) for each head
        p_pred = self.phase_picker_p(features_p.reshape(B * S, half, Tf))
        s_pred = self.phase_picker_s(features_s.reshape(B * S, Cf - half, Tf))
        
        # Reshape phase predictions back to (B, S, Tf)
        p_pred = p_pred.view(B, S, Tf)
        s_pred = s_pred.view(B, S, Tf)
        
        # Apply mask to phase predictions (zero out invalid stations)
        mask_expanded = mask.unsqueeze(-1).float()  # (B, S, 1)
        p_pred = p_pred * mask_expanded
        s_pred = s_pred * mask_expanded
        
        # Event detection from aggregated features
        event_pred = self.event_detector(agg_features)  # (B, Tf)
        
        # Depth estimation from aggregated features
        depth_pred = self.depth_estimator(agg_features)  # (B,)
        
        return p_pred, s_pred, event_pred, depth_pred

In [ ]:
# ============================================
# BLOCK 6: LOSS FUNCTION (with gradient clipping and numerical stability)
# ============================================

class EQNetPlusLoss(nn.Module):
    """Multi-task loss for EQNet+ with numerical stability improvements"""
    def __init__(self, config):
        super().__init__()
        self.lambda_p = config.lambda_p
        self.lambda_s = config.lambda_s
        self.lambda_event = config.lambda_event
        self.lambda_depth = config.lambda_depth
        self.bce = nn.BCEWithLogitsLoss(reduction='none')
        self.mse = nn.MSELoss()
        
    def forward(self, p_pred, s_pred, event_pred, depth_pred, batch):
        """
        batch contains:
            p_label: (B, S, T)  - T is original time length (2000)
            s_label: (B, S, T)
            event_time_label: (B, T)
            event_depth: (B, 1) - normalized
            mask: (B, S)
        """
        B, S, T_orig = batch['p_label'].shape
        _, _, T_feat = p_pred.shape  # T_feat = 63 for 2000 input
        
        # Move everything to the same device
        device = p_pred.device
        
        # Simple downsampling by taking every 32nd sample
        p_label_down = batch['p_label'][:, :, ::32].to(device)  # (B, S, T_feat)
        s_label_down = batch['s_label'][:, :, ::32].to(device)  # (B, S, T_feat)
        event_label_down = batch['event_time_label'][:, ::32].to(device)  # (B, T_feat)
        
        # Ensure shapes match exactly
        if p_label_down.shape[-1] > T_feat:
            p_label_down = p_label_down[:, :, :T_feat]
            s_label_down = s_label_down[:, :, :T_feat]
            event_label_down = event_label_down[:, :T_feat]
        elif p_label_down.shape[-1] < T_feat:
            # Pad if needed
            pad_size = T_feat - p_label_down.shape[-1]
            p_label_down = F.pad(p_label_down, (0, pad_size))
            s_label_down = F.pad(s_label_down, (0, pad_size))
            event_label_down = F.pad(event_label_down, (0, pad_size))
        
        # Phase losses (masked)
        mask = batch['mask'].unsqueeze(-1).to(device)  # (B, S, 1)
        
        # Add small epsilon for numerical stability
        eps = 1e-7
        
        # Apply mask to predictions and labels
        p_loss_unmasked = self.bce(p_pred, p_label_down)
        p_loss = (p_loss_unmasked * mask).sum() / (mask.sum() + eps)
        
        s_loss_unmasked = self.bce(s_pred, s_label_down)
        s_loss = (s_loss_unmasked * mask).sum() / (mask.sum() + eps)
        
        # Event detection loss
        event_loss = self.bce(event_pred, event_label_down).mean()
        
        # Depth estimation loss (on normalized values)
        depth_target = batch['event_depth'].to(device).squeeze()
        depth_loss = self.mse(depth_pred, depth_target)
        
        # Combined loss
        total = (self.lambda_p * p_loss + 
                 self.lambda_s * s_loss + 
                 self.lambda_event * event_loss + 
                 self.lambda_depth * depth_loss)
        
        # Check for NaN and return 0 if NaN (shouldn't happen with these improvements)
        if torch.isnan(total):
            print("Warning: NaN loss detected, returning 0")
            total = torch.tensor(0.0, device=device, requires_grad=True)
        
        losses = {
            'p_loss': p_loss.item() if not torch.isnan(p_loss) else 0,
            's_loss': s_loss.item() if not torch.isnan(s_loss) else 0,
            'event_loss': event_loss.item() if not torch.isnan(event_loss) else 0,
            'depth_loss': depth_loss.item() if not torch.isnan(depth_loss) else 0,
            'total': total.item() if not torch.isnan(total) else 0
        }
        return total, losses

In [ ]:
# ============================================
# BLOCK 10: MAIN TRAINING LOOP (COMPLETELY FIXED)
# ============================================

def train_one_epoch(model, loader, optimizer, criterion, scaler, config, epoch):
    model.train()
    total_loss = 0
    valid_batches = 0
    pbar = tqdm(loader, desc=f'Epoch {epoch+1} Train')
    
    for batch_idx, batch in enumerate(pbar):
        try:
            # Skip if batch is None
            if batch is None:
                continue
                
            # Move to device
            for k in batch:
                if isinstance(batch[k], torch.Tensor):
                    batch[k] = batch[k].to(config.device)
            
            optimizer.zero_grad()
            
            if config.mixed_precision:
                with autocast():
                    p_pred, s_pred, event_pred, depth_pred = model(
                        batch['waveforms'], batch['station_coords'], batch['mask']
                    )
                    
                    # Debug: print shapes for first batch of first epoch
                    if batch_idx == 0 and epoch == 0:
                        print(f"\nBatch {batch_idx} shapes:")
                        print(f"  waveforms: {batch['waveforms'].shape}")
                        print(f"  p_label: {batch['p_label'].shape}")
                        print(f"  mask: {batch['mask'].shape}")
                        print(f"  p_pred: {p_pred.shape}")
                        print(f"  s_pred: {s_pred.shape}")
                        print(f"  event_pred: {event_pred.shape}")
                        print(f"  depth_pred: {depth_pred.shape}")
                    
                    loss, losses = criterion(p_pred, s_pred, event_pred, depth_pred, batch)
                
                # Check if loss is valid
                if torch.isnan(loss) or torch.isinf(loss):
                    print(f"\nWarning: Invalid loss detected in batch {batch_idx}, skipping...")
                    # Skip this batch entirely - don't call backward or step
                    continue
                
                # Scale loss and backward
                scaler.scale(loss).backward()
                
                # Unscale gradients
                scaler.unscale_(optimizer)
                
                # Clip gradients
                torch.nn.utils.clip_grad_norm_(model.parameters(), config.gradient_clip)
                
                # Step optimizer and update scaler
                scaler.step(optimizer)
                scaler.update()
                
                total_loss += loss.item()
                valid_batches += 1
                
                # Update progress bar
                pbar.set_postfix({
                    'loss': f"{loss.item():.4f}",
                    'p': f"{losses['p_loss']:.4f}",
                    's': f"{losses['s_loss']:.4f}",
                    'ev': f"{losses['event_loss']:.4f}",
                    'dep': f"{losses['depth_loss']:.4f}"
                })
                
            else:
                p_pred, s_pred, event_pred, depth_pred = model(
                    batch['waveforms'], batch['station_coords'], batch['mask']
                )
                loss, losses = criterion(p_pred, s_pred, event_pred, depth_pred, batch)
                
                if torch.isnan(loss) or torch.isinf(loss):
                    print(f"\nWarning: Invalid loss detected in batch {batch_idx}, skipping...")
                    continue
                    
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), config.gradient_clip)
                optimizer.step()
                
                total_loss += loss.item()
                valid_batches += 1
                
                # Update progress bar
                pbar.set_postfix({
                    'loss': f"{loss.item():.4f}",
                    'p': f"{losses['p_loss']:.4f}",
                    's': f"{losses['s_loss']:.4f}",
                    'ev': f"{losses['event_loss']:.4f}",
                    'dep': f"{losses['depth_loss']:.4f}"
                })
            
        except Exception as e:
            print(f"\nError in batch {batch_idx}: {e}")
            import traceback
            traceback.print_exc()
            # Reset optimizer state if needed
            optimizer.zero_grad()
            continue
        
        # Clear cache periodically
        if batch_idx % 100 == 0:
            torch.cuda.empty_cache()
            gc.collect()
    
    avg_loss = total_loss / max(1, valid_batches)
    return avg_loss, scaler


def validate(model, loader, criterion, config):
    model.eval()
    total_loss = 0
    all_losses = {'p_loss': 0, 's_loss': 0, 'event_loss': 0, 'depth_loss': 0}
    n_batches = 0
    
    with torch.no_grad():
        for batch in tqdm(loader, desc='Validating'):
            try:
                if batch is None:
                    continue
                    
                for k in batch:
                    if isinstance(batch[k], torch.Tensor):
                        batch[k] = batch[k].to(config.device)
                
                p_pred, s_pred, event_pred, depth_pred = model(
                    batch['waveforms'], batch['station_coords'], batch['mask']
                )
                loss, losses = criterion(p_pred, s_pred, event_pred, depth_pred, batch)
                
                # Skip if loss is invalid
                if torch.isnan(loss) or torch.isinf(loss):
                    continue
                
                total_loss += loss.item()
                for k in all_losses:
                    all_losses[k] += losses[k]
                n_batches += 1
                
            except Exception as e:
                print(f"Validation error: {e}")
                continue
    
    if n_batches == 0:
        return float('inf'), all_losses
    
    return total_loss / n_batches, {k: v/n_batches for k, v in all_losses.items()}

In [ ]:
# ============================================
# BLOCK 8: EVALUATION FUNCTIONS
# ============================================

# ============================================
# BLOCK 8: EVALUATION FUNCTIONS (updated with sigmoid)
# ============================================

def evaluate_phase_picking(model, test_loader, config):
    """Compute picking residuals and precision/recall."""
    model.eval()
    p_residuals = []
    s_residuals = []
    p_correct = 0
    s_correct = 0
    total_p = 0
    total_s = 0
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc='Evaluating phase picking'):
            for k in batch:
                if isinstance(batch[k], torch.Tensor):
                    batch[k] = batch[k].to(config.device)
            
            p_pred, s_pred, _, _ = model(batch['waveforms'], batch['station_coords'], batch['mask'])
            
            # Apply sigmoid to get probabilities
            p_pred = torch.sigmoid(p_pred)
            s_pred = torch.sigmoid(s_pred)
            
            # Compare to labels (only where mask is 1 and label exists)
            mask = batch['mask'].bool()
            
            # Time dimension factor (backbone downsamples by 32)
            time_factor = 32 / config.sampling_rate  # seconds per time step in feature space
            
            for i in range(p_pred.shape[0]):  # batch
                for j in range(p_pred.shape[1]):  # station
                    if not mask[i, j]:
                        continue
                    
                    # P phase
                    if batch['p_label'][i, j].max() > 0.5:
                        total_p += 1
                        pred_peak = p_pred[i, j].argmax().item()
                        true_peak = batch['p_label'][i, j].argmax().item()
                        # Convert to seconds (accounting for downsampling)
                        resid = (pred_peak - true_peak) * time_factor
                        p_residuals.append(resid)
                        if abs(resid) < 0.5:
                            p_correct += 1
                    
                    # S phase
                    if batch['s_label'][i, j].max() > 0.5:
                        total_s += 1
                        pred_peak = s_pred[i, j].argmax().item()
                        true_peak = batch['s_label'][i, j].argmax().item()
                        resid = (pred_peak - true_peak) * time_factor
                        s_residuals.append(resid)
                        if abs(resid) < 0.5:
                            s_correct += 1
    
    # Calculate metrics
    p_precision = p_correct / total_p if total_p > 0 else 0
    s_precision = s_correct / total_s if total_s > 0 else 0
    
    print("\n" + "="*50)
    print("PHASE PICKING EVALUATION")
    print("="*50)
    print(f"P-phase: Precision within 0.5s = {p_precision:.4f} ({p_correct}/{total_p})")
    if p_residuals:
        print(f"P residuals: mean={np.mean(p_residuals):.4f}s, std={np.std(p_residuals):.4f}s")
    
    print(f"\nS-phase: Precision within 0.5s = {s_precision:.4f} ({s_correct}/{total_s})")
    if s_residuals:
        print(f"S residuals: mean={np.mean(s_residuals):.4f}s, std={np.std(s_residuals):.4f}s")
    
    return {
        'p_precision': p_precision,
        'p_residuals': p_residuals,
        's_precision': s_precision,
        's_residuals': s_residuals
    }


def evaluate_event_detection(model, test_loader, config):
    """Compute event detection metrics."""
    model.eval()
    y_true = []
    y_score = []
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc='Evaluating event detection'):
            for k in batch:
                if isinstance(batch[k], torch.Tensor):
                    batch[k] = batch[k].to(config.device)
            
            _, _, event_pred, _ = model(batch['waveforms'], batch['station_coords'], batch['mask'])
            
            # Apply sigmoid to get probabilities
            event_pred = torch.sigmoid(event_pred)
            
            # Ground truth: event exists if Gaussian label has peak > 0.5
            true = (batch['event_time_label'] > 0.5).any(dim=1).float()  # (B,)
            # Prediction: max score over time
            pred_score = event_pred.max(dim=1)[0]  # (B,)
            
            y_true.extend(true.cpu().numpy())
            y_score.extend(pred_score.cpu().numpy())
    
    # Calculate precision-recall
    precision, recall, thresholds = precision_recall_curve(y_true, y_score)
    ap = average_precision_score(y_true, y_score)
    
    print("\n" + "="*50)
    print("EVENT DETECTION EVALUATION")
    print("="*50)
    print(f"Average Precision (AP) = {ap:.4f}")
    print(f"Number of test windows: {len(y_true)}")
    print(f"Positive events: {sum(y_true)}")
    
    # Find optimal threshold (F1)
    f1_scores = 2 * precision * recall / (precision + recall + 1e-10)
    best_idx = np.argmax(f1_scores[:-1])
    best_thresh = thresholds[best_idx]
    best_f1 = f1_scores[best_idx]
    
    print(f"Best threshold: {best_thresh:.4f} (F1 = {best_f1:.4f})")
    print(f"Precision at best: {precision[best_idx]:.4f}")
    print(f"Recall at best: {recall[best_idx]:.4f}")
    
    return {
        'ap': ap,
        'precision': precision,
        'recall': recall,
        'thresholds': thresholds,
        'best_threshold': best_thresh,
        'best_f1': best_f1
    }


def evaluate_depth(model, test_loader, config):
    """Compute depth estimation error."""
    model.eval()
    errors = []
    predictions = []
    targets = []
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc='Evaluating depth'):
            for k in batch:
                if isinstance(batch[k], torch.Tensor):
                    batch[k] = batch[k].to(config.device)
            
            _, _, _, depth_pred = model(batch['waveforms'], batch['station_coords'], batch['mask'])
            true_depth = batch['event_depth']
            
            errors.extend((depth_pred - true_depth).cpu().numpy())
            predictions.extend(depth_pred.cpu().numpy())
            targets.extend(true_depth.cpu().numpy())
    
    errors = np.array(errors)
    rmse = np.sqrt(np.mean(errors**2))
    mae = np.mean(np.abs(errors))
    
    print("\n" + "="*50)
    print("DEPTH ESTIMATION EVALUATION (NOVEL)")
    print("="*50)
    print(f"RMSE = {rmse:.4f} km")
    print(f"MAE = {mae:.4f} km")
    print(f"Mean error = {np.mean(errors):.4f} km")
    print(f"Std error = {np.std(errors):.4f} km")
    
    return {
        'rmse': rmse,
        'mae': mae,
        'errors': errors,
        'predictions': predictions,
        'targets': targets
    }

In [ ]:
# ============================================
# BLOCK 9: VISUALIZATION FUNCTIONS
# ============================================

def plot_training_history(train_losses, val_losses, save_path=None):
    """Plot training and validation loss curves."""
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    plt.plot(train_losses, label='Train', linewidth=2)
    plt.plot(val_losses, label='Validation', linewidth=2)
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training History')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.subplot(1, 2, 2)
    plt.plot(train_losses, label='Train', linewidth=2)
    plt.plot(val_losses, label='Validation', linewidth=2)
    plt.yscale('log')
    plt.xlabel('Epoch')
    plt.ylabel('Loss (log scale)')
    plt.title('Training History (Log Scale)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


def plot_phase_picking_example(model, dataset, idx=0, save_path=None):
    """Plot phase picking results for a single example."""
    model.eval()
    
    # Get sample
    sample = dataset[idx]
    waveforms = sample['waveforms'].unsqueeze(0).to(Config.device)
    station_coords = sample['station_coords'].unsqueeze(0).to(Config.device)
    mask = sample['mask'].unsqueeze(0).to(Config.device)
    
    with torch.no_grad():
        p_pred, s_pred, _, _ = model(waveforms, station_coords, mask)
    
    # Move to CPU
    p_pred = p_pred[0].cpu().numpy()
    s_pred = s_pred[0].cpu().numpy()
    p_true = sample['p_label'].numpy()
    s_true = sample['s_label'].numpy()
    mask = sample['mask'].numpy()
    
    # Upsample predictions to original time scale (simple repeat)
    time_factor = 32
    T_orig = sample['waveforms'].shape[-1]
    T_feat = p_pred.shape[-1]
    
    p_pred_upsampled = np.repeat(p_pred, time_factor, axis=1)[:, :T_orig]
    s_pred_upsampled = np.repeat(s_pred, time_factor, axis=1)[:, :T_orig]
    
    # Plot first few stations
    n_plot = min(5, len(mask[mask > 0]))
    fig, axes = plt.subplots(n_plot, 1, figsize=(15, 3*n_plot))
    if n_plot == 1:
        axes = [axes]
    
    time_axis = np.arange(T_orig) / Config.sampling_rate
    
    plot_idx = 0
    for i in range(len(mask)):
        if mask[i] == 0:
            continue
        if plot_idx >= n_plot:
            break
            
        ax = axes[plot_idx]
        
        # Plot Z component waveform
        waveform = sample['waveforms'][i, 2].numpy()  # Z component
        ax.plot(time_axis, waveform / np.max(np.abs(waveform)), 'k', alpha=0.5, label='Waveform')
        
        # Plot P predictions and true
        ax.plot(time_axis, p_pred_upsampled[i], 'r-', linewidth=2, label='P pred')
        ax.plot(time_axis, p_true[i], 'r--', linewidth=2, label='P true')
        
        # Plot S predictions and true
        ax.plot(time_axis, s_pred_upsampled[i], 'b-', linewidth=2, label='S pred')
        ax.plot(time_axis, s_true[i], 'b--', linewidth=2, label='S true')
        
        ax.set_xlim([20, 40])  # Zoom in
        ax.set_ylabel(f'Station {i}')
        ax.legend(loc='upper right')
        ax.grid(True, alpha=0.3)
        
        plot_idx += 1
    
    axes[-1].set_xlabel('Time (s)')
    plt.suptitle('Phase Picking Example', fontsize=14)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


def plot_precision_recall_curve(precision, recall, ap, save_path=None):
    """Plot precision-recall curve."""
    plt.figure(figsize=(8, 6))
    plt.plot(recall, precision, linewidth=2, label=f'AP = {ap:.4f}')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title('Precision-Recall Curve')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.xlim([0, 1])
    plt.ylim([0, 1])
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


def plot_depth_estimates(targets, predictions, save_path=None):
    """Plot depth estimation results."""
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    plt.scatter(targets, predictions, alpha=0.5, s=10)
    min_val = min(min(targets), min(predictions))
    max_val = max(max(targets), max(predictions))
    plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Ideal')
    plt.xlabel('True Depth (km)')
    plt.ylabel('Predicted Depth (km)')
    plt.title('Depth Estimation')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.subplot(1, 2, 2)
    errors = np.array(predictions) - np.array(targets)
    plt.hist(errors, bins=50, alpha=0.7, edgecolor='black')
    plt.xlabel('Error (km)')
    plt.ylabel('Count')
    plt.title(f'Depth Error Distribution (RMSE={np.sqrt(np.mean(errors**2)):.2f} km)')
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# ============================================
# BLOCK 10.5: UPDATED MAIN FUNCTION
# ============================================

def main():
    print("="*60)
    print("EQNet+: Attention-based Multi-Station Earthquake Detection")
    print("="*60)
    print(f"Device: {Config.device}")
    print(f"Batch size: {Config.batch_size}")
    print(f"Epochs: {Config.epochs}")
    print(f"Max stations per event: {Config.max_stations_per_event}")
    print("="*60)
    
    # Create datasets
    print("\nLoading datasets...")
    try:
        train_dataset = STEADMultiStationDataset(Config.csv_path, Config.hdf5_path, 'train', Config)
        val_dataset = STEADMultiStationDataset(Config.csv_path, Config.hdf5_path, 'val', Config)
    except Exception as e:
        print(f"Error loading datasets: {e}")
        return None, [], []
    
    # Create data loaders
    train_loader = DataLoader(
        train_dataset, 
        batch_size=Config.batch_size, 
        shuffle=True,
        num_workers=0,
        pin_memory=False,
        drop_last=True
    )
    val_loader = DataLoader(
        val_dataset, 
        batch_size=Config.batch_size, 
        shuffle=False,
        num_workers=0,
        pin_memory=False,
        drop_last=True
    )
    
    print(f"Train batches: {len(train_loader)}")
    print(f"Val batches: {len(val_loader)}")
    
    # Create model
    print("\nCreating model...")
    model = EQNetPlus(Config).to(Config.device)
    
    # Loss function
    criterion = EQNetPlusLoss(Config)
    
    # Optimizer and scheduler
    optimizer = AdamW(model.parameters(), lr=Config.lr, weight_decay=Config.weight_decay)
    scheduler = CosineAnnealingLR(optimizer, T_max=Config.epochs)
    
    # Mixed precision scaler
    scaler = GradScaler() if Config.mixed_precision else None
    
    # Training history
    train_losses = []
    val_losses = []
    best_val_loss = float('inf')
    
    print("\nStarting training...")
    print("="*60)
    
    for epoch in range(Config.epochs):
        print(f"\nEpoch {epoch+1}/{Config.epochs}")
        print("-"*40)
        
        # Train
        train_loss, scaler = train_one_epoch(model, train_loader, optimizer, criterion, scaler, Config, epoch)
        train_losses.append(train_loss)
        
        # Validate
        val_loss, val_loss_details = validate(model, val_loader, criterion, Config)
        val_losses.append(val_loss)
        
        # Scheduler step
        scheduler.step()
        
        # Print epoch summary
        print(f"\nEpoch {epoch+1} Summary:")
        print(f"  Train Loss: {train_loss:.6f}")
        print(f"  Val Loss: {val_loss:.6f}")
        if val_loss != float('inf'):
            print(f"  Val Details: P={val_loss_details['p_loss']:.4f}, "
                  f"S={val_loss_details['s_loss']:.4f}, "
                  f"Event={val_loss_details['event_loss']:.4f}, "
                  f"Depth={val_loss_details['depth_loss']:.4f}")
        print(f"  Learning Rate: {scheduler.get_last_lr()[0]:.2e}")
        
        # Save checkpoint
        checkpoint_path = os.path.join(Config.checkpoint_dir, f'checkpoint_epoch{epoch+1}.pth')
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'train_loss': train_loss,
            'val_loss': val_loss,
            'config': Config
        }, checkpoint_path)
        
        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_path = os.path.join(Config.checkpoint_dir, 'best_model.pth')
            torch.save(model.state_dict(), best_path)
            print(f"  *** New best model saved! (val_loss: {val_loss:.6f}) ***")
        
        # Clear cache
        torch.cuda.empty_cache()
        gc.collect()
    
    # Save final model
    final_path = os.path.join(Config.checkpoint_dir, 'final_model.pth')
    torch.save(model.state_dict(), final_path)
    print(f"\nTraining complete. Final model saved to {final_path}")
    
    # Plot training history if we have losses
    if train_losses and val_losses and val_losses[0] != float('inf'):
        try:
            plot_training_history(
                train_losses, 
                val_losses, 
                save_path=os.path.join(Config.output_dir, 'training_history.png')
            )
        except Exception as e:
            print(f"Error plotting: {e}")
    
    return model, train_losses, val_losses

In [ ]:
# ============================================
# BLOCK 15: DEBUG INITIAL LOSS VALUES
# ============================================

print("Testing initial loss values without training...")

# Create model and criterion
model = EQNetPlus(Config).to(Config.device)
criterion = EQNetPlusLoss(Config)

# Get a few batches
train_dataset = STEADMultiStationDataset(Config.csv_path, Config.hdf5_path, 'train', Config)
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, num_workers=0)

model.eval()
with torch.no_grad():
    for i, batch in enumerate(train_loader):
        if i >= 3:  # Check first 3 batches
            break
            
        print(f"\n{'='*50}")
        print(f"Batch {i}:")
        print(f"{'='*50}")
        
        # Move to device
        for k in batch:
            if isinstance(batch[k], torch.Tensor):
                batch[k] = batch[k].to(Config.device)
        
        # Forward pass
        p_pred, s_pred, event_pred, depth_pred = model(
            batch['waveforms'], batch['station_coords'], batch['mask']
        )
        
        print(f"Predictions stats:")
        print(f"  p_pred: min={p_pred.min():.3f}, max={p_pred.max():.3f}, mean={p_pred.mean():.3f}")
        print(f"  s_pred: min={s_pred.min():.3f}, max={s_pred.max():.3f}, mean={s_pred.mean():.3f}")
        print(f"  event_pred: min={event_pred.min():.3f}, max={event_pred.max():.3f}, mean={event_pred.mean():.3f}")
        print(f"  depth_pred: min={depth_pred.min():.3f}, max={depth_pred.max():.3f}, mean={depth_pred.mean():.3f}")
        
        print(f"\nLabel stats:")
        print(f"  p_label: min={batch['p_label'].min():.3f}, max={batch['p_label'].max():.3f}")
        print(f"  s_label: min={batch['s_label'].min():.3f}, max={batch['s_label'].max():.3f}")
        print(f"  event_time_label: min={batch['event_time_label'].min():.3f}, max={batch['event_time_label'].max():.3f}")
        print(f"  event_depth: {batch['event_depth'].cpu().numpy()}")
        
        # Compute loss
        loss, losses = criterion(p_pred, s_pred, event_pred, depth_pred, batch)
        
        print(f"\nLoss values:")
        print(f"  total: {loss.item():.6f}")
        print(f"  p_loss: {losses['p_loss']:.6f}")
        print(f"  s_loss: {losses['s_loss']:.6f}")
        print(f"  event_loss: {losses['event_loss']:.6f}")
        print(f"  depth_loss: {losses['depth_loss']:.6f}")
        
        # Check for NaN
        if torch.isnan(loss):
            print("  ❌ NaN detected in total loss!")
        else:
            print("  ✅ Loss is valid")

In [ ]:
# ============================================
# BLOCK 11: COMPREHENSIVE EVALUATION
# ============================================

def evaluate_model(model_path=None):
    """Run comprehensive evaluation on test set."""
    print("="*60)
    print("EQNet+ Comprehensive Evaluation")
    print("="*60)
    
    # Load model
    if model_path and os.path.exists(model_path):
        print(f"\nLoading model from {model_path}")
        model = EQNetPlus(Config).to(Config.device)
        model.load_state_dict(torch.load(model_path, map_location=Config.device))
    else:
        print("\nNo model path provided, using untrained model")
        model = EQNetPlus(Config).to(Config.device)
    
    # Create test dataset
    print("\nLoading test dataset...")
    test_dataset = STEADMultiStationDataset(Config.csv_path, Config.hdf5_path, 'test', Config)
    test_loader = DataLoader(
        test_dataset, 
        batch_size=Config.batch_size, 
        shuffle=False,
        num_workers=Config.num_workers, 
        pin_memory=True
    )
    print(f"Test batches: {len(test_loader)}")
    
    # Run evaluations
    print("\n" + "="*60)
    print("RUNNING EVALUATIONS")
    print("="*60)
    
    # Phase picking evaluation
    phase_results = evaluate_phase_picking(model, test_loader, Config)
    
    # Event detection evaluation
    event_results = evaluate_event_detection(model, test_loader, Config)
    
    # Depth estimation evaluation (NOVEL)
    depth_results = evaluate_depth(model, test_loader, Config)
    
    # Plot results
    print("\nGenerating plots...")
    
    # Plot precision-recall curve
    plot_precision_recall_curve(
        event_results['precision'],
        event_results['recall'],
        event_results['ap'],
        save_path=os.path.join(Config.output_dir, 'pr_curve.png')
    )
    
    # Plot depth estimates
    plot_depth_estimates(
        depth_results['targets'],
        depth_results['predictions'],
        save_path=os.path.join(Config.output_dir, 'depth_estimation.png')
    )
    
    # Plot phase picking example
    plot_phase_picking_example(
        model, 
        test_dataset, 
        idx=0,
        save_path=os.path.join(Config.output_dir, 'phase_picking_example.png')
    )
    
    # Summary
    print("\n" + "="*60)
    print("EVALUATION SUMMARY")
    print("="*60)
    print(f"\nPHASE PICKING:")
    print(f"  P-phase precision (0.5s): {phase_results['p_precision']:.4f}")
    print(f"  S-phase precision (0.5s): {phase_results['s_precision']:.4f}")
    
    print(f"\nEVENT DETECTION:")
    print(f"  Average Precision: {event_results['ap']:.4f}")
    print(f"  Best F1: {event_results['best_f1']:.4f}")
    print(f"  Best threshold: {event_results['best_threshold']:.4f}")
    
    print(f"\nDEPTH ESTIMATION (NOVEL):")
    print(f"  RMSE: {depth_results['rmse']:.4f} km")
    print(f"  MAE: {depth_results['mae']:.4f} km")
    
    # Save results to file
    results = {
        'phase_picking': {
            'p_precision': float(phase_results['p_precision']),
            'p_mean_residual': float(np.mean(phase_results['p_residuals'])) if phase_results['p_residuals'] else None,
            'p_std_residual': float(np.std(phase_results['p_residuals'])) if phase_results['p_residuals'] else None,
            's_precision': float(phase_results['s_precision']),
            's_mean_residual': float(np.mean(phase_results['s_residuals'])) if phase_results['s_residuals'] else None,
            's_std_residual': float(np.std(phase_results['s_residuals'])) if phase_results['s_residuals'] else None,
        },
        'event_detection': {
            'ap': float(event_results['ap']),
            'best_f1': float(event_results['best_f1']),
            'best_threshold': float(event_results['best_threshold']),
        },
        'depth_estimation': {
            'rmse': float(depth_results['rmse']),
            'mae': float(depth_results['mae']),
        }
    }
    
    import json
    results_path = os.path.join(Config.output_dir, 'evaluation_results.json')
    with open(results_path, 'w') as f:
        json.dump(results, f, indent=2)
    print(f"\nResults saved to {results_path}")
    
    return results

In [ ]:
# ============================================
# BLOCK 12: INFERENCE FOR CONTINUOUS DATA
# ============================================

def infer_continuous(model, waveforms, station_coords, station_mask, config):
    """
    Run inference on continuous waveform data.
    
    Args:
        model: Trained EQNetPlus model
        waveforms: (n_stations, 3, total_samples) tensor
        station_coords: (n_stations, 2) tensor of [lat, lon]
        station_mask: (n_stations,) boolean tensor
        config: Config object
    
    Returns:
        List of detected events with time, depth, and phase picks
    """
    model.eval()
    
    stride = config.window_samples // 2  # 50% overlap
    n_stations, _, total_len = waveforms.shape
    T = config.window_samples
    Tf = T // 32  # after backbone downsampling
    
    # Prepare sliding windows
    windows = []
    start_times = []
    
    for start in range(0, total_len - T + 1, stride):
        windows.append(waveforms[:, :, start:start+T])
        start_times.append(start / config.sampling_rate)
    
    if not windows:
        return []
    
    windows = torch.stack(windows)  # (n_windows, n_stations, 3, T)
    n_windows = windows.shape[0]
    
    # Pad stations if needed
    if n_stations < config.max_stations_per_event:
        pad = config.max_stations_per_event - n_stations
        windows = F.pad(windows, (0, 0, 0, 0, 0, 0, 0, pad))
        station_coords_padded = F.pad(station_coords, (0, 0, 0, pad))
        station_mask_padded = F.pad(station_mask, (0, pad))
    else:
        station_coords_padded = station_coords
        station_mask_padded = station_mask
    
    # Process in batches
    batch_size = config.batch_size
    detections = []
    
    for i in range(0, n_windows, batch_size):
        batch_waves = windows[i:i+batch_size].to(config.device)
        batch_coords = station_coords_padded.unsqueeze(0).expand(len(batch_waves), -1, -1).to(config.device)
        batch_mask = station_mask_padded.unsqueeze(0).expand(len(batch_waves), -1).to(config.device)
        
        with torch.no_grad():
            p_pred, s_pred, event_pred, depth_pred = model(batch_waves, batch_coords, batch_mask)
        
        # Process each window
        for b in range(len(batch_waves)):
            event_curve = event_pred[b].cpu().numpy()
            
            # Find peaks
            peaks, properties = find_peaks(
                event_curve, 
                height=config.detection_threshold,
                distance=config.nms_window_sec * config.sampling_rate / 32
            )
            
            for pk_idx, pk in enumerate(peaks):
                # Convert peak index to time
                time_in_window = pk * 32 / config.sampling_rate
                abs_time = start_times[i+b] + time_in_window
                depth = depth_pred[b].item()
                confidence = properties['peak_heights'][pk_idx]
                
                # Get phase picks for this window at the detected time
                # Find corresponding indices in phase predictions
                # (simplified: take argmax near event time)
                window_phase_picks = {}
                for station_idx in range(n_stations):
                    if station_mask[station_idx]:
                        # Find P pick within [-5, 10]s of event time
                        p_curve = p_pred[b, station_idx].cpu().numpy()
                        s_curve = s_pred[b, station_idx].cpu().numpy()
                        
                        # Search window around event time (in feature space)
                        search_radius = int(10 * config.sampling_rate / 32)  # 10 seconds
                        event_feat_idx = pk
                        start_idx = max(0, event_feat_idx - search_radius)
                        end_idx = min(len(p_curve), event_feat_idx + search_radius)
                        
                        if start_idx < end_idx:
                            p_peak = start_idx + np.argmax(p_curve[start_idx:end_idx])
                            s_peak = start_idx + np.argmax(s_curve[start_idx:end_idx])
                            
                            window_phase_picks[station_idx] = {
                                'p_time': start_times[i+b] + p_peak * 32 / config.sampling_rate,
                                'p_confidence': float(p_curve[p_peak]),
                                's_time': start_times[i+b] + s_peak * 32 / config.sampling_rate,
                                's_confidence': float(s_curve[s_peak])
                            }
                
                detections.append({
                    'time': abs_time,
                    'depth_km': depth,
                    'confidence': float(confidence),
                    'window_idx': i+b,
                    'peak_idx': int(pk),
                    'phase_picks': window_phase_picks
                })
    
    # Non-maximum suppression (merge nearby detections)
    detections.sort(key=lambda x: x['time'])
    merged = []
    
    for d in detections:
        if not merged or d['time'] - merged[-1]['time'] > config.nms_window_sec:
            merged.append(d)
        else:
            # Keep the one with higher confidence
            if d['confidence'] > merged[-1]['confidence']:
                merged[-1] = d
    
    return merged


def demo_inference():
    """Demonstrate inference on a small continuous segment."""
    print("\n" + "="*60)
    print("INFERENCE DEMO")
    print("="*60)
    
    # Load a trained model
    model_path = os.path.join(Config.checkpoint_dir, 'best_model.pth')
    if not os.path.exists(model_path):
        print("No trained model found. Please train first.")
        return
    
    model = EQNetPlus(Config).to(Config.device)
    model.load_state_dict(torch.load(model_path, map_location=Config.device))
    print(f"Model loaded from {model_path}")
    
    # Create a small continuous test segment from test dataset
    test_dataset = STEADMultiStationDataset(Config.csv_path, Config.hdf5_path, 'test', Config)
    test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False, num_workers=0)
    
    try:
        batch = next(iter(test_loader))
        print("Batch keys:", batch.keys())
        print("waveforms shape:", batch['waveforms'].shape)
        print("station_coords shape:", batch['station_coords'].shape)
        print("mask shape:", batch['mask'].shape)
        print("p_label shape:", batch['p_label'].shape)
        print("s_label shape:", batch['s_label'].shape)
        print("event_time_label shape:", batch['event_time_label'].shape)
        print("event_depth shape:", batch['event_depth'].shape)
        print("n_stations:", batch['n_stations'])
        
        # Also check min/max values
        print(f"waveforms min: {batch['waveforms'].min():.3f}, max: {batch['waveforms'].max():.3f}")
        
    except Exception as e:
        print(f"Error in dataset: {e}")
        import traceback
        traceback.print_exc()
    
    # Take first few events and concatenate to form continuous data
    continuous_length = Config.window_samples * 3  # 3 windows
    n_stations = Config.max_stations_per_event
    
    # Get station coordinates from first event
    sample = test_dataset[0]
    station_coords = sample['station_coords'][:sample['n_stations'], :]  # (n, 2)
    station_mask = torch.ones(sample['n_stations'])
    
    # Build continuous waveform by concatenating windows
    waveforms_list = []
    for i in range(3):
        sample = test_dataset[i]
        waveforms_list.append(sample['waveforms'][:sample['n_stations'], :, :])
    
    # Simple concatenation with gap
    gap = torch.zeros(sample['n_stations'], 3, Config.sampling_rate * 10)  # 10s gap
    continuous_waveforms = torch.cat([
        waveforms_list[0], 
        gap, 
        waveforms_list[1], 
        gap, 
        waveforms_list[2]
    ], dim=2)
    
    print(f"Continuous data shape: {continuous_waveforms.shape}")
    print(f"Duration: {continuous_waveforms.shape[-1] / Config.sampling_rate:.1f} seconds")
    
    # Run inference
    detections = infer_continuous(
        model, 
        continuous_waveforms, 
        station_coords, 
        station_mask, 
        Config
    )
    
    print(f"\nDetected {len(detections)} events:")
    for i, d in enumerate(detections):
        print(f"  Event {i+1}: time={d['time']:.2f}s, depth={d['depth_km']:.2f}km, confidence={d['confidence']:.3f}")
    
    return detections

In [ ]:
# ============================================
# BLOCK 14: DEBUGGING - Check dataset shapes
# ============================================

print("="*60)
print("DEBUGGING: Checking dataset shapes")
print("="*60)

# Create a small test dataset
test_dataset = STEADMultiStationDataset(Config.csv_path, Config.hdf5_path, 'train', Config)
test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False, num_workers=0)

# Get first batch
try:
    batch = next(iter(test_loader))
    
    print("\nBatch keys:", batch.keys())
    print("\nShape information:")
    print(f"waveforms shape: {batch['waveforms'].shape}")
    print(f"station_coords shape: {batch['station_coords'].shape}")
    print(f"mask shape: {batch['mask'].shape}")
    print(f"p_label shape: {batch['p_label'].shape}")
    print(f"s_label shape: {batch['s_label'].shape}")
    print(f"event_time_label shape: {batch['event_time_label'].shape}")
    print(f"event_depth shape: {batch['event_depth'].shape}")
    print(f"n_stations: {batch['n_stations']}")
    
    print("\nValue ranges:")
    print(f"waveforms - min: {batch['waveforms'].min():.3f}, max: {batch['waveforms'].max():.3f}")
    print(f"mask - unique values: {torch.unique(batch['mask'])}")
    
    # Test the model forward pass with this batch
    print("\n" + "="*60)
    print("Testing model forward pass with this batch")
    print("="*60)
    
    # Create model
    model = EQNetPlus(Config).to(Config.device)
    model.eval()
    
    # Move batch to device
    for k in batch:
        if isinstance(batch[k], torch.Tensor):
            batch[k] = batch[k].to(Config.device)
    
    # Try forward pass
    with torch.no_grad():
        try:
            p_pred, s_pred, event_pred, depth_pred = model(
                batch['waveforms'], 
                batch['station_coords'], 
                batch['mask']
            )
            print("\n✓ Forward pass successful!")
            print(f"p_pred shape: {p_pred.shape}")
            print(f"s_pred shape: {s_pred.shape}")
            print(f"event_pred shape: {event_pred.shape}")
            print(f"depth_pred shape: {depth_pred.shape}")
        except Exception as e:
            print(f"\n✗ Forward pass failed with error:")
            print(f"Error type: {type(e).__name__}")
            print(f"Error message: {str(e)}")
            
            # Print shapes at each step for debugging
            print("\nDetailed shape tracing:")
            B, S, C, T = batch['waveforms'].shape
            print(f"Input waveforms: ({B}, {S}, {C}, {T})")
            
            # Reshape for backbone
            w = batch['waveforms'].view(B * S, C, T)
            print(f"After reshape for backbone: {w.shape}")
            
            # Try backbone
            try:
                features = model.backbone(w)
                print(f"After backbone: {features.shape}")
            except Exception as e_back:
                print(f"Backbone error: {e_back}")
                
except Exception as e:
    print(f"Error creating dataset or loader: {e}")
    import traceback
    traceback.print_exc()

print("\n" + "="*60)
print("Debugging complete")
print("="*60)

In [ ]:
# ============================================
# BLOCK 13: MAIN EXECUTION
# ============================================

# Run the complete pipeline
if __name__ == "__main__":
    # Clear cache before starting
    torch.cuda.empty_cache()
    gc.collect()
    
    print("\n" + "="*70)
    print("EQNet+: Attention-based Multi-Station Earthquake Detection with Depth Estimation")
    print("="*70)
    print("\nNOVEL CONTRIBUTIONS:")
    print("1. Cross-station attention mechanism for adaptive multi-station fusion")
    print("2. Direct depth estimation head for complete hypocenter location")
    print("3. End-to-end joint optimization of all tasks")
    print("="*70)
    
    # Choose what to run (reduce for memory constraints)
    run_training = True
    run_evaluation = False  # Set to False initially to save memory
    run_inference_demo = False
    
    # Training
    if run_training:
        print("\n" + "="*70)
        print("STAGE 1: TRAINING")
        print("="*70)
        try:
            model, train_losses, val_losses = main()
        except Exception as e:
            print(f"Training error: {e}")
            import traceback
            traceback.print_exc()
    
    # Evaluation (commented out for now to save memory)
    if run_evaluation:
        print("\n" + "="*70)
        print("STAGE 2: COMPREHENSIVE EVALUATION")
        print("="*70)
        try:
            model_path = os.path.join(Config.checkpoint_dir, 'best_model.pth') if run_training else None
            results = evaluate_model(model_path)
        except Exception as e:
            print(f"Evaluation error: {e}")
            traceback.print_exc()
    
    # Inference demo
    if run_inference_demo:
        print("\n" + "="*70)
        print("STAGE 3: INFERENCE DEMONSTRATION")
        print("="*70)
        try:
            detections = demo_inference()
        except Exception as e:
            print(f"Inference error: {e}")
            traceback.print_exc()
    
    print("\n" + "="*70)
    print("Pipeline complete! Results saved to:", Config.output_dir)
    print("="*70)
    
    # Final cleanup
    torch.cuda.empty_cache()
    gc.collect()